# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.

Let's enumerate all `RecordSet` objects and, for each, show their `@id`, name, and fields.

_Note: Entity references use the `@id` (identifier) fields, as best practice with Croissant._

In [ ]:
# List all record sets and summarize their fields

record_sets = list(dataset.record_sets)
print(f"Total record sets found: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - id: {field.id}, name: {field.name}, dataType: {field.data_type}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We will extract the data from all record sets found.

In [ ]:
# Extract data from each record set using @id

dataframes = {}

for rs in record_sets:
    rs_id = rs.id
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records for RecordSet {rs_id}.")
        else:
            print(f"No records found for RecordSet {rs_id}.")
    except Exception as e:
        print(f"Could not load records for RecordSet {rs_id}: {e}")

# Show columns for the first loaded record set
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns for RecordSet {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like filtering, removing outliers, transforming distributions, or grouping.

In [ ]:
# Choose a RecordSet and a numeric field for EDA

if dataframes:
    record_set_id = first_rs_id  # Use first loaded RecordSet
    df = dataframes[record_set_id]

    # Try to auto-detect a numeric field for demonstration
    numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Using numeric field for EDA: {numeric_field}")
        threshold = df[numeric_field].mean() if not df[numeric_field].isnull().all() else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, norm_col]].head())

        # Optional: try grouping by a likely categorical field
        candidate_groups = [col for col in df.columns if df[col].nunique() < min(10, len(df)) and col != numeric_field]
        if candidate_groups:
            group_field = candidate_groups[0]
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field} (showing means):")
            print(grouped_df.head())
        else:
            print("No categorical field suitable for grouping detected.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No DataFrames loaded to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We plot a histogram of the first available numeric field in the first record set (if available).

In [ ]:
import matplotlib.pyplot as plt

if dataframes and numeric_fields:
    plt.figure(figsize=(6, 4))
    df[numeric_field].hist(bins=30, alpha=0.7)
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.title(f'Distribution of {numeric_field}')
    plt.show()
else:
    print("No numeric field found to visualize.")

## 6. Conclusion
In this notebook, we've demonstrated how to:
- Load Croissant metadata and data with `mlcroissant`, referencing all entities by `@id`.
- Explore the dataset's record sets and fields.
- Extract data into pandas DataFrames for analysis.
- Perform basic exploratory data analysis and visualization.

You can adapt this template to deeper analyses or other datasets shared in Croissant format.